In [1]:
import polars as pl

In [2]:
df = pl.read_parquet("../../data/results.parquet")

# Cohen's d using the pooled standard deviation of the two groups
# groups with a single sample have a null std, which contributes 0 to the pooled variance
n_pos = pl.col("count_positive")
n_neg = pl.col("count_negative")
pooled_std = (
    ((n_pos - 1) * pl.col("std_positive").fill_null(0) ** 2 + (n_neg - 1) * pl.col("std_negative").fill_null(0) ** 2)
    / (n_pos + n_neg - 2)
).sqrt()
df = df.with_columns(((pl.col("mean_positive") - pl.col("mean_negative")) / pooled_std).alias("cohens_d"))

# prune findings with a negligible effect size (|d| < 0.2), or where d is undefined (zero pooled std or both groups n=1)
MIN_EFFECT_SIZE = 0.2
n_before = df.height
df = df.filter(pl.col("cohens_d").is_finite() & (pl.col("cohens_d").abs() >= MIN_EFFECT_SIZE))
print(f"kept {df.height}/{n_before} findings with |cohen's d| >= {MIN_EFFECT_SIZE}")

kept 60723/85326 findings with |cohen's d| >= 0.2


In [3]:
df

bioproject,col_name,col_value,statistic,pvalue,mean_positive,std_positive,count_positive,mean_negative,std_negative,count_negative,cohens_d
str,list[str],list[str],f64,f64,f64,f64,i32,f64,f64,i32,f64
"""PRJNA448599""","[""source_name_attrib"", ""phenotype_attrib""]","[""Single non-hemogenic endothelial cells"", ""Cdh5+ Gfi1- Gfi1b- c-kit-""]",9.0,0.232967,0.139858,0.009861,3,0.463729,0.433505,12,-0.812144
"""PRJNA448599""","[""source_name_attrib"", ""phenotype_attrib""]","[""Single EHT cells"", ""Cdh5+ Gfi1+ Gfi1b- c-kit+""]",56.0,0.000311,0.705771,0.421236,7,0.13049,0.052028,8,1.992598
"""PRJNA448599""","[""source_name_attrib""]","[""Single Intra-aortic hematopoietic cluster cells""]",1.0,0.038095,0.077477,0.0702,2,0.448413,0.415903,13,-0.927199
"""PRJNA448599""","[""source_name_attrib"", ""phenotype_attrib""]","[""Single hemogenic endothelial cells"", ""Cdh5+ Gfi1+ Gfi1b- c-kit-""]",11.0,0.364835,0.156465,0.054394,3,0.459577,0.436172,12,-0.754411
"""PRJNA448599""","[""strain_attrib"", ""genotype_attrib""]","[""C57BL/6"", ""WT""]",1.0,0.038095,0.077477,0.0702,2,0.448413,0.415903,13,-0.927199
…,…,…,…,…,…,…,…,…,…,…,…
"""PRJNA793775""","[""time_of_last_follow_up_in_months_attrib""]","[""23""]",38.0,0.384673,0.592455,0.867302,10,0.427397,0.330637,10,0.251487
"""PRJNA793775""","[""time_of_last_follow_up_in_months_attrib""]","[""21""]",14.0,0.673684,0.243846,0.204838,2,0.53949,0.672604,18,-0.451067
"""PRJNA612785""","[""tissue_attrib""]","[""HPV16 positive cervical epithelium tissues""]",3.0,0.7,0.022746,0.007433,3,0.03232,0.020681,3,-0.616108


In [4]:
def process_row(row):
    col_names = row["col_name"]
    values = row["col_value"]
    attributes = [f"{col}={val}" for col, val in zip(col_names, values)]
    results = f"Samples with the following attributes: {', '.join(attributes)} had a mean concentration of totiviridae of {row['mean_positive']} and samples with different attributes had a mean concentration of totiviridae of {row['mean_negative']}."
    return results

# add a new column to the dataframe with the processed results
df = df.with_columns(
    pl.struct(["col_name", "col_value", "mean_positive", "mean_negative"])
    .map_elements(process_row, return_dtype=pl.String)
    .alias("processed_result")
)

In [5]:
def build_hypothesis(row):
    return f"Analyzing SRA metadata and totiviridae abundance, we found the following statistical finding: {row['processed_result']} Please come up with a hypothesis that could explain this finding."

df = df.with_columns(
    pl.struct(["processed_result"])
    .map_elements(build_hypothesis, return_dtype=pl.String)
    .alias("prompt")
)

In [6]:
# add in columns for response, input tokens, output tokens, and cost
df = df.with_columns(
    pl.lit("").alias("response"),
    pl.lit(0).alias("input_tokens"),
    pl.lit(0).alias("output_tokens"),
    pl.lit(0.0).alias("cost")
)

In [7]:
from openai import AsyncOpenAI
from openai.providers import bedrock
from dotenv import load_dotenv
import asyncio
import os
load_dotenv()

# retries with exponential backoff on rate limits and transient errors
client = AsyncOpenAI(max_retries=5)

In [8]:
MODEL = "openai.gpt-6-luna"

# price in $ per 1M tokens
INPUT = 0.22
OUTPUT = 1.32

# maximum number of requests in flight at once
MAX_CONCURRENT = 80

async def query_hypothesis(prompt):
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
        except Exception as e:
            # leave the row empty so it is retried on the next run
            print(f"request failed: {e}")
            return {"response": "", "input_tokens": 0, "output_tokens": 0, "cost": 0.0}
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (output_tokens * OUTPUT + input_tokens * INPUT) / 1000000
    return {
        "response": response.choices[0].message.content or "",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
    }

In [9]:
CHECKPOINT_PATH = "../../data/hypotheses_checkpoint.parquet"
CHECKPOINT_EVERY = 1000

# number of rows to query this run, set to None to run on all remaining rows
N_ROWS = 10000

semaphore = asyncio.Semaphore(MAX_CONCURRENT)
result_cols = ["response", "input_tokens", "output_tokens", "cost"]
if "row_idx" not in df.columns:
    df = df.with_row_index("row_idx")

# resume from the checkpoint if one exists
if os.path.exists(CHECKPOINT_PATH):
    df = df.update(pl.read_parquet(CHECKPOINT_PATH), on="row_idx")

todo = df.filter(pl.col("response") == "")
if N_ROWS is not None:
    todo = todo.head(N_ROWS)
print(f"{df.height - df.filter(pl.col('response') == '').height} rows already done, querying {todo.height} rows")

for start in range(0, todo.height, CHECKPOINT_EVERY):
    chunk = todo.slice(start, CHECKPOINT_EVERY)
    results = await asyncio.gather(*(query_hypothesis(prompt) for prompt in chunk["prompt"]))

    results_df = pl.DataFrame(results, schema={col: df.schema[col] for col in result_cols}).with_columns(chunk["row_idx"])
    df = df.update(results_df, on="row_idx")

    # save every completed row so far, failed rows are left out and retried next run
    df.filter(pl.col("response") != "").select(["row_idx", *result_cols]).write_parquet(CHECKPOINT_PATH)
    print(f"checkpointed {start + chunk.height}/{todo.height} rows, total cost so far: ${df['cost'].sum():.4f}")

df = df.drop("row_idx")

48498 rows already done, querying 10000 rows
checkpointed 1000/10000 rows, total cost so far: $26.9878
checkpointed 2000/10000 rows, total cost so far: $27.5408
checkpointed 3000/10000 rows, total cost so far: $28.0855
checkpointed 4000/10000 rows, total cost so far: $28.6112
checkpointed 5000/10000 rows, total cost so far: $29.1499
checkpointed 6000/10000 rows, total cost so far: $29.6755
checkpointed 7000/10000 rows, total cost so far: $30.1999
checkpointed 8000/10000 rows, total cost so far: $30.7284
checkpointed 9000/10000 rows, total cost so far: $31.2807
checkpointed 10000/10000 rows, total cost so far: $31.8154


In [11]:
df.write_parquet("../../data/results_with_hypotheses.parquet")

In [12]:
# sum up the total cost of all queries
total_cost = df["cost"].sum()
total_cost

11.142021659999997